This version reads the first N rows from `../dimension-reduction/data/1000_sampled_classified_embeddings.geojson` (which are randomly sampled global geolocations generated by `../dimension-reduction/data-processing.ipynb` and already have the umap 2d values).

It is an improved version of us-similarity-grid.


In [4]:
import ee
import json
import math
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import base64
import math

from shapely.geometry import Point

# from tqdm import tqdm
# from umap import UMAP
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

# ---- USER INPUTS ----
# Run gen_rand_coords.ipynb first to generate N random coordinates
N = 25  # number of random points to sample
EMBED_PIXELS = 64  # patch width (px)
EMBED_SCALE = 10  # m/px (AlphaEarth native)
YEAR = 2024
SEED = 42

GEOJSON_PATH = "../dimension-reduction/data/1000_sampled_classified_embeddings.geojson"
OUTPUT_DIR = f"output/{N}_{EMBED_PIXELS}_{EMBED_SCALE}_{YEAR}_global"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Read precomputed embeddings and UMAP coords from GeoJSON, arrange into grid

print("Loading GeoJSON with precomputed embeddings and UMAP coordinates ...")

# ---- READ GEOJSON ----
with open(GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

# Parse features into DataFrame
features = geojson_data["features"][:N]  # Take first N features
data = []

for feat in features:
    props = feat["properties"]
    coords = feat["geometry"]["coordinates"]  # [lon, lat]

    data.append(
        {
            "lon": coords[0],
            "lat": coords[1],
            "umap_2d_x": props["umap_2d_x"],
            "umap_2d_y": props["umap_2d_y"],
            "classification": props.get("classification"),
            "embedding_b64": props.get("embedding"),  # base64-encoded
        }
    )

gdf = gpd.GeoDataFrame(
    data, geometry=[Point(d["lon"], d["lat"]) for d in data], crs="EPSG:4326"
)

print(f"✔ Loaded {len(gdf)} points from {GEOJSON_PATH}")
print(f"  Columns: {list(gdf.columns)}\n")

# ---- EXTRACT UMAP COORDINATES ----
print("Extracting UMAP 2D coordinates ...")
umap_2d = gdf[["umap_2d_x", "umap_2d_y"]].values.astype(np.float32)
print(f"UMAP shape: {umap_2d.shape}")
print(f"  X range: [{umap_2d[:, 0].min():.2f}, {umap_2d[:, 0].max():.2f}]")
print(f"  Y range: [{umap_2d[:, 1].min():.2f}, {umap_2d[:, 1].max():.2f}]\n")

# ---- ARRANGE INTO GRID ----
print("Arranging points into grid ...")
n = len(gdf)
grid_side = int(math.ceil(math.sqrt(n)))
grid_x = np.linspace(0, 1, grid_side)
grid_y = np.linspace(0, 1, grid_side)
grid_coords = np.array(np.meshgrid(grid_x, grid_y)).T.reshape(-1, 2)[:n]

# Hungarian algorithm for minimal assignment cost
print("  Running Hungarian algorithm ...")
cost = cdist(umap_2d, grid_coords)
r_idx, c_idx = linear_sum_assignment(cost)
print("✔ Grid arrangement complete.\n")

# ---- ADD GRID COORDINATES TO GDF ----
print("Adding grid coordinates to GeoDataFrame ...")

# Normalized grid coordinates (0..1)
assigned_coords = grid_coords[c_idx]
gdf.loc[r_idx, "grid_x_norm"] = assigned_coords[:, 0]
gdf.loc[r_idx, "grid_y_norm"] = assigned_coords[:, 1]

# Integer grid positions
grid_cols = (assigned_coords[:, 0] * (grid_side - 1)).round().astype(int)
grid_rows = (assigned_coords[:, 1] * (grid_side - 1)).round().astype(int)
gdf.loc[r_idx, "grid_x"] = grid_cols
gdf.loc[r_idx, "grid_y"] = grid_rows

# ---- COMPUTE PIXEL POSITIONS (TOP-LEFT ORIGIN) ----
print("Computing pixel positions ...")
gdf["pixel_x"] = gdf["grid_x"] * EMBED_PIXELS
gdf["pixel_y"] = gdf["grid_y"] * EMBED_PIXELS

print(f"Grid size: {grid_side}×{grid_side}")
print(f"Output image size: {grid_side * EMBED_PIXELS}×{grid_side * EMBED_PIXELS} px\n")

# ---- SAVE ----
print("Saving GeoDataFrame to disk ...")
geojson_path = f"{OUTPUT_DIR}/{N}_{EMBED_PIXELS}_{EMBED_SCALE}_{YEAR}_data.geojson"
csv_path = f"{OUTPUT_DIR}/{N}_{EMBED_PIXELS}_{EMBED_SCALE}_{YEAR}_data.csv"

gdf.to_file(geojson_path, driver="GeoJSON")
gdf.drop(columns="geometry").to_csv(csv_path, index=False)

print(f"✔ Saved to:\n  {geojson_path}\n  {csv_path}")

# Store for next cell
valid_gdf = gdf.copy()
valid_points = [(row["lat"], row["lon"]) for _, row in gdf.iterrows()]

Loading GeoJSON with precomputed embeddings and UMAP coordinates ...
✔ Loaded 25 points from ../dimension-reduction/data/1000_sampled_classified_embeddings.geojson
  Columns: ['lon', 'lat', 'umap_2d_x', 'umap_2d_y', 'classification', 'embedding_b64', 'geometry']

Extracting UMAP 2D coordinates ...
UMAP shape: (25, 2)
  X range: [-48.50, 49.32]
  Y range: [-49.46, 42.39]

Arranging points into grid ...
  Running Hungarian algorithm ...
✔ Grid arrangement complete.

Adding grid coordinates to GeoDataFrame ...
Computing pixel positions ...
Grid size: 5×5
Output image size: 320×320 px

Saving GeoDataFrame to disk ...
✔ Saved to:
  output/25_64_10_2024_global/25_64_10_2024_data.geojson
  output/25_64_10_2024_global/25_64_10_2024_data.csv


In [10]:
# Download satellite patches and create grid image (batched + low-cloud filter)

import requests
from io import BytesIO
from PIL import Image
import time

ee.Initialize(project="gsapp-map")

# ---- USER INPUTS ----
TILE_PX = EMBED_PIXELS
side_meters_patch = EMBED_PIXELS * EMBED_SCALE
CLOUD_THRESHOLD = 10  # max cloud % per image
BATCH_SIZE = 20  # fetch this many thumbnails per batch

print("Preparing satellite thumbnail grid ...")

# ---- Stricter Sentinel-2 composite: low-cloud imagery ----
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filter(ee.Filter.calendarRange(YEAR - 20, YEAR, "year"))
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_THRESHOLD))
)
s2_median = s2.select(["B4", "B3", "B2"]).median()

print(f"Cloud threshold: {CLOUD_THRESHOLD}%\n")

# ---- Output grid image size ----
grid_side = int(np.ceil(np.sqrt(len(valid_points))))
img_size = (grid_side * TILE_PX, grid_side * TILE_PX)
sat_grid_img = Image.new("RGB", img_size, (0, 0, 0))


# ---- Helper to fetch thumbnail ----
def fetch_satellite_thumbnail(center_lon, center_lat, side_meters, out_px):
    geom = ee.Geometry.Point([center_lon, center_lat]).buffer(side_meters / 2).bounds()
    try:
        url = s2_median.getThumbURL(
            {
                "region": geom.getInfo(),
                "dimensions": f"{out_px}x{out_px}",
                "format": "png",
                "min": 0,
                "max": 3000,
                "bands": ["B4", "B3", "B2"],
            }
        )
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content)).convert("RGB")
    except Exception as e:
        print(f"    ✖ Failed at ({center_lat:.5f},{center_lon:.5f}): {e}")
        return Image.new("RGB", (out_px, out_px), (50, 50, 50))


# ---- Batch download and paste into grid ----
valid_gdf_reindex = gdf.loc[r_idx].reset_index(drop=True)
print(f"Downloading {len(valid_gdf_reindex)} thumbnails in batches of {BATCH_SIZE} ...")

total_rows = len(valid_gdf_reindex)
for batch_start in range(0, total_rows, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_rows)
    batch_rows = valid_gdf_reindex.iloc[batch_start:batch_end]

    print(
        f"\nBatch {batch_start // BATCH_SIZE + 1}: fetching indices {batch_start}–{batch_end - 1}..."
    )
    batch_start_time = time.time()

    for local_idx, (i, row) in enumerate(batch_rows.iterrows()):
        lat, lon = row["lat"], row["lon"]
        gx, gy = int(row["grid_x"]), int(row["grid_y"])
        left = gx * TILE_PX
        upper = gy * TILE_PX

        thumb = fetch_satellite_thumbnail(lon, lat, side_meters_patch, TILE_PX)
        sat_grid_img.paste(thumb, (left, upper))

        print(
            f"  ✔ [{batch_start + local_idx + 1}/{total_rows}] grid=({gx},{gy}) pixel=({left},{upper})"
        )

        # ---- Save individual patch ----
        os.makedirs(
            f"{OUTPUT_DIR}/{N}_{TILE_PX}_{EMBED_SCALE}_{YEAR}_sat_patches",
            exist_ok=True,
        )
        patch_path = os.path.join(
            f"{OUTPUT_DIR}/{N}_{TILE_PX}_{EMBED_SCALE}_{YEAR}_sat_patches",
            f"patch_{batch_start + local_idx}_lat{lat:.5f}_lon{lon:.5f}.png",
        )
        thumb.save(patch_path)

    batch_time = time.time() - batch_start_time
    print(f"  Batch completed in {batch_time:.1f}s")

# ---- Save output ----
sat_out_path = f"{OUTPUT_DIR}/{N}_{TILE_PX}_{EMBED_SCALE}_{YEAR}_sentinel_grid.png"
sat_grid_img.save(sat_out_path)
print(f"\n✔ Saved satellite thumbnail grid to: {sat_out_path}")

Preparing satellite thumbnail grid ...
Cloud threshold: 10%


Batch 1: fetching indices 0–19...
  ✔ [1/25] grid=(1,3) pixel=(64,192)
  ✔ [2/25] grid=(4,3) pixel=(256,192)
  ✔ [3/25] grid=(4,4) pixel=(256,256)
  ✔ [4/25] grid=(3,2) pixel=(192,128)
  ✔ [5/25] grid=(0,3) pixel=(0,192)
  ✔ [6/25] grid=(4,2) pixel=(256,128)
  ✔ [7/25] grid=(3,4) pixel=(192,256)
  ✔ [8/25] grid=(2,3) pixel=(128,192)
  ✔ [9/25] grid=(3,1) pixel=(192,64)
  ✔ [10/25] grid=(0,2) pixel=(0,128)
  ✔ [11/25] grid=(4,1) pixel=(256,64)
  ✔ [12/25] grid=(1,1) pixel=(64,64)
  ✔ [13/25] grid=(0,1) pixel=(0,64)
  ✔ [14/25] grid=(0,0) pixel=(0,0)
  ✔ [15/25] grid=(2,4) pixel=(128,256)
  ✔ [16/25] grid=(2,1) pixel=(128,64)
  ✔ [17/25] grid=(4,0) pixel=(256,0)
  ✔ [18/25] grid=(2,2) pixel=(128,128)
  ✔ [19/25] grid=(1,0) pixel=(64,0)
  ✔ [20/25] grid=(3,0) pixel=(192,0)
  Batch completed in 34.6s

Batch 2: fetching indices 20–24...
  ✔ [21/25] grid=(0,4) pixel=(0,256)
  ✔ [22/25] grid=(1,2) pixel=(64,128)
  ✔ [23/25] grid=(2